In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
from scipy.stats import norm  # 用于高斯函数D(T)计算
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import CosineAnnealingLR

# 设备配置（CPU/GPU）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)  # 固定随机种子，保证可复现
np.random.seed(42)

In [2]:
# -------------------------- 几何参数（文档2.1节）--------------------------
d0 = 0.05  # 内管直径（m），文档d0=50mm
d1 = 0.13  # 外壳直径（m），文档d1=130mm
r0 = d0 / 2  # 内管半径
r1 = d1 / 2  # 外壳半径

# -------------------------- 材料热物性参数（文档表1）--------------------------
# PCM（石蜡）
rho_s = 880.0    # 固相密度 (kg/m³)
rho_l = 760.0    # 液相密度 (kg/m³)
cp_s = 2180.0    # 固相定压比热容 (J/(kg·K))
cp_l = 2390.0    # 液相定压比热容 (J/(kg·K))
lambda_s = 0.4   # 固相导热系数 (W/(m·K))
lambda_l = 0.15  # 液相导热系数 (W/(m·K))
mu_l = 0.001     # 液相粘度 (kg/(m·s))
L = 255000.0     # 相变潜热 (J/kg)，文档L=255kJ/kg转换为J/kg
Tpc = 316.15     # 相变温度 (K)
DeltaT = 6.0     # 相变温度区间 (K)

# 高导热材料（铜）
rho_Cu = 8960.0  # 密度 (kg/m³)
lambda_Cu = 400.0  # 导热系数 (W/(m·K))
cp_Cu = 385.0    # 定压比热容 (J/(kg·K))
mu_Cu = 1e10     # 铜为固体，粘度取极大值抑制流动

# 外壳材料（铝）- 仅用于边界，此处拓扑优化设计域为PCM+铜，铝不参与设计
cp_Al = 879.0    # 定压比热容 (J/(kg·K))

# -------------------------- 物理模型参数（文档2.2节）--------------------------
Am = 1e5         # 糊状区常数（文献常用值，文档未明确）
epsilon = 0.001  # 避免分母为0（文档方程4）
xi = 1.0         # 粘度函数常数（文档方程10）
alpha = 1e-4     # PCM体胀系数 (1/K)，自然对流计算需
g = 9.81         # 重力加速度 (m/s²)

# -------------------------- 拓扑优化参数（文档2.3节）--------------------------
phi_total = 0.3  # 高导热材料体积比约束（预设，文档未明确，工程常用0.3）
case = 3         # 优化目标选择：1=平均温度，2=温度均方差，3=多目标
w1, w2, w3 = 1.0, 1.0, 1.0  # 多目标权重（Case3用）

# -------------------------- 训练超参数 --------------------------
N_mass = 10000   # 质量守恒方程采样点数量
N_mom = 10000    # 动量方程采样点数量
N_heat = 10000   # 传热方程采样点数量
N_IC = 8000      # 初始条件采样点数量
N_BC1 = 3000     # 内管壁边界采样点数量
N_BC2 = 3000     # 外壳边界采样点数量
N_rho = 10000    # 拓扑设计变量采样点数量
N_vol = 10000    # 体积比约束采样点数量
N_obj = 10000    # 优化目标采样点数量

# 损失项权重（对应思路3.1节）
lambda1 = 1e3    # PDE损失权重
lambda2 = 1.5e4    # IC/BC损失权重
lambda3 = 5.0   # 拓扑约束损失权重
lambda4 = 1.0    # 优化目标损失权重

In [3]:
class ResidualBlock(nn.Module):
    """残差块：缓解深层网络梯度消失，提升表达能力"""
    def __init__(self, dim):
        super(ResidualBlock, self).__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.tanh = nn.Tanh()  # 保持与原网络一致的激活函数
    
    def forward(self, x):
        residual = x  # 残差连接：保留输入特征
        out = self.tanh(self.fc1(x))
        out = self.fc2(out)
        return self.tanh(out + residual)  # 残差相加后激活
        
class TopoPINN(nn.Module):
    def __init__(self, input_dim=3, output_dim=6, hidden_layers=6, hidden_dim=256):
        super(TopoPINN, self).__init__()
        # 输入层：3维（x,y,τ）→ hidden_dim维
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        # 隐藏层：6个残差块（提升表达能力，匹配复杂耦合关系）
        self.hidden_layers = nn.ModuleList([ResidualBlock(hidden_dim) for _ in range(hidden_layers)])
        # 输出层：hidden_dim维→6维（ux, uy, p, T, φ, ρx）
        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x: [batch_size, 3]，对应(x,y,τ)
        out = self.input_layer(x)
        out = torch.tanh(out)  # 输入层后激活
        for layer in self.hidden_layers:
            out = layer(out)
        # 输出层无激活，后续对物理量进行归一化约束
        out = self.output_layer(out)
        # 分离输出并归一化（平衡物理量量级，提升训练稳定性）
        ux = out[:, 0:1] * 1e-3  # 速度量级：1e-3 m/s（符合自然对流速度）
        uy = out[:, 1:2] * 1e-3
        p = out[:, 2:3] * 1e3    # 压力量级：1e3 Pa（流动压力）
        T = out[:, 3:4] * 50 + 300  # 温度量级：250-350 K（50*[-1,1]+300）
        phi = torch.sigmoid(out[:, 4:5])  # 液相率∈[0,1]（sigmoid约束更平滑）
        rho_x = torch.sigmoid(out[:, 5:6])  # 拓扑变量∈[0,1]
        return ux, uy, p, T, phi, rho_x

In [4]:
def compute_thermo_props(phi, rho_x, T):
    """
    计算混合材料的热物性参数（PCM+铜），对应文档方程8-13
    输入：phi(T)（液相率）、rho_x（拓扑设计变量）、T（温度）
    输出：rho_total, lambda_total, mu_total, cp_total, a（热扩散率）
    """
    # -------------------------- 1. 液相率φ的物理约束（思路2.1）--------------------------
    # 虽然后续有损失项约束，但此处先做数值裁剪，避免计算溢出
    phi = torch.clamp(phi, 1e-6, 1.0 - 1e-6)  # φ∈[1e-6, 0.999]

    # -------------------------- 2. 密度ρ_total（文档方程8 + ρx加权）--------------------------
    # PCM密度：ρ_PCM = ρ_s + (ρ_l - ρ_s)*φ
    rho_PCM = rho_s + (rho_l - rho_s) * phi
    # 混合密度：ρ_total = ρx*ρ_Cu + (1-ρx)*ρ_PCM（铜占比ρx，PCM占比1-ρx）
    rho_total = rho_x * rho_Cu + (1.0 - rho_x) * rho_PCM

    # -------------------------- 3. 导热系数λ_total（文档方程9 + ρx加权）--------------------------
    lambda_PCM = lambda_s + (lambda_l - lambda_s) * phi
    lambda_total = rho_x * lambda_Cu + (1.0 - rho_x) * lambda_PCM

    # -------------------------- 4. 粘度mu_total（文档方程10-11 + ρx加权）--------------------------
    # 糊状区源项S_t(T)（文档方程4）
    S_t = Am * ((1.0 - phi) ** 2) / (phi ** 3 + epsilon + 1e-10)  # 新增1e-10，增强稳定性
    # PCM粘度（文档方程10）
    mu_PCM = mu_l + S_t * xi + 1e-10  # 避免mu_PCM=0
    # 混合粘度：铜为固体，mu_Cu取1e10抑制流动
    mu_total = rho_x * mu_Cu + (1.0 - rho_x) * mu_PCM
    # 运动粘度ν = μ/ρ（文档方程11）
    nu = mu_total / rho_total

    # -------------------------- 5. 定压比热容cp_total（文档方程12-13 + ρx加权）--------------------------
    # D(T)：高斯函数（文档方程13），平滑化相变潜热
    sigma = DeltaT / 4.0  # 标准差
    D_T = torch.exp(-((T - Tpc) ** 2) / (sigma ** 2)) / (torch.sqrt(torch.tensor(np.pi)) * sigma)
    D_T = torch.clamp(D_T, 0.0, 1e3)  # 裁剪D_T
    # PCM比热容（含潜热，文档方程12）
    cp_PCM = cp_s + phi * (cp_l - cp_s) + L * D_T
    # 混合比热容：铜占比ρx，PCM占比1-ρx
    cp_total = rho_x * cp_Cu + (1.0 - rho_x) * cp_PCM

    # -------------------------- 6. 热扩散率a（文档方程7）--------------------------
    a = lambda_total / (rho_total * cp_total + 1e-10)

    return rho_total, lambda_total, mu_total, nu, cp_total, a, S_t

In [5]:
def sample_collocation_points(N):
    """采样设计域内配点（x,y,τ）+ 面积元权重，用于PDE损失计算"""
    # 1. 时间τ采样：τ∈[0, 1e5s]（文档2.4.2节计算时间），归一化到[0,1]便于训练
    tau = np.random.uniform(0.0, 1e5, size=(N, 1))  # [N,1]
    tau_norm = tau / 1e5  # 归一化到[0,1]

    # 2. 环形域（x,y）采样：极坐标转换（避免采样到域外）
    r = np.random.uniform(r0, r1, size=(N, 1))  # 半径∈[r0, r1]
    theta = np.random.uniform(0.0, 2*np.pi, size=(N, 1))  # 角度∈[0,2π]
    x = r * np.cos(theta)  # x坐标
    y = r * np.sin(theta)  # y坐标

    # 3. 输入归一化：x,y∈[-r1, r1]→归一化到[-1,1]，提升训练稳定性
    x_norm = x / r1
    y_norm = y / r1

    # 4. 面积元权重：环形域dA=r*dr*dθ，假设dr/dθ均匀，权重≈r（归一化后）
    weight = r / np.mean(r)  # 归一化权重（总和为N，便于平均）

    # 合并为输入张量：[N, 3] = (x_norm, y_norm, tau_norm)
    points = np.hstack([x_norm, y_norm, tau_norm])
    return (torch.tensor(points, dtype=torch.float32).to(device),
            torch.tensor(weight, dtype=torch.float32).to(device))  # 返回（配点，权重）

def adaptive_sample(model, N, prev_points, prev_residuals, residual_threshold=0.1):
    """
    自适应采样：聚焦高残差区域，提升关键区域精度
    prev_points: 上一轮配点 [N,3]
    prev_residuals: 上一轮传热残差 [N,1]
    residual_threshold: 高残差阈值（取前20%）
    """
    # 1. 筛选高残差点（前20%）
    high_residual_idx = torch.topk(prev_residuals.squeeze(), int(N*0.2))[1]
    high_residual_points = prev_points[high_residual_idx]
    
    # 2. 生成新采样点（80%）
    new_points, new_weights = sample_collocation_points(int(N*0.8))
    
    # 3. 合并自适应采样点（新点+高残差点）
    adaptive_points = torch.cat([new_points, high_residual_points], dim=0)
    # 补全权重（高残差点权重沿用之前的权重，或用1.0填充）
    high_residual_weights = prev_weights[high_residual_idx]  # 提取高残差点的原权重
    adaptive_weights = torch.cat([new_weights, high_residual_weights], dim=0)    
    return adaptive_points, adaptive_weights

def sample_initial_condition(N):
    """采样初始条件点（τ=0, x,y∈设计域）+ 面积元权重"""
    # 时间τ=0，归一化后为0
    tau_norm = np.zeros((N, 1))

    # 环形域（x,y）采样（同配点采样逻辑）
    r = np.random.uniform(r0, r1, size=(N, 1))
    theta = np.random.uniform(0.0, 2*np.pi, size=(N, 1))
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    x_norm = x / r1
    y_norm = y / r1

    # 面积元权重：环形域dA=r*dr*dθ，假设dr/dθ均匀，权重≈r（归一化后）
    weight = r / np.mean(r)

    points = np.hstack([x_norm, y_norm, tau_norm])
    return (torch.tensor(points, dtype=torch.float32).to(device),
            torch.tensor(weight, dtype=torch.float32).to(device))  # 返回（初始点，权重）

def sample_boundary(N1, N2):
    """采样边界点：N1=内管壁（Dirichlet），N2=外壳（Neumann），对应思路4.3"""
    # 1. 内管壁（x²+y²=r0²）
    theta1 = np.random.uniform(0.0, 2*np.pi, size=(N1, 1))
    x1 = r0 * np.cos(theta1)
    y1 = r0 * np.sin(theta1)
    x1_norm = x1 / r1
    y1_norm = y1 / r1
    tau1_norm = np.random.uniform(0.0, 1.0, size=(N1, 1))  # 时间∈[0,1]（归一化后）
    bc1_points = np.hstack([x1_norm, y1_norm, tau1_norm])

    # 2. 外壳（x²+y²=r1²）
    theta2 = np.random.uniform(0.0, 2*np.pi, size=(N2, 1))
    x2 = r1 * np.cos(theta2)
    y2 = r1 * np.sin(theta2)
    x2_norm = x2 / r1
    y2_norm = y2 / r1
    tau2_norm = np.random.uniform(0.0, 1.0, size=(N2, 1))
    bc2_points = np.hstack([x2_norm, y2_norm, tau2_norm])

    return (torch.tensor(bc1_points, dtype=torch.float32).to(device),
            torch.tensor(bc2_points, dtype=torch.float32).to(device))

In [6]:
def compute_loss(model, collocation_points, collocation_weights, ic_points, ic_weights, bc1_points, bc2_points, heat_storage=True):
    """
    计算总损失L_total = λ1*L_PDE + λ2*L_IC/BC + λ3*L_topo-constraint + λ4*L_objective
    heat_storage: True=储热过程，False=释热过程（对应文档2.2.3节初始/边界条件）
    """
    # -------------------------- 1. 预处理：获取各类采样点的网络输出 --------------------------
    # 配点输出（用于PDE损失）
    model.train()
    x_col = collocation_points.requires_grad_(True)  # 需计算梯度，开启autograd
    ux_col, uy_col, p_col, T_col, phi_col, rho_x_col = model(x_col)
    rho_x_col = torch.clamp(rho_x_col, 1e-6, 1.0 - 1e-6)
    T_col = torch.clamp(T_col, 285.0, 365.0)  # 约束温度
    # 计算热物性参数
    rho_total, lambda_total, mu_total, nu_col, cp_total, a_col, S_t = compute_thermo_props(phi_col, rho_x_col, T_col)

    # 初始条件点输出（用于IC损失）
    x_ic = ic_points.requires_grad_(True)
    ux_ic, uy_ic, p_ic, T_ic, phi_ic, rho_x_ic = model(x_ic)

    # 边界点输出（用于BC损失）
    x_bc1 = bc1_points.requires_grad_(True)  # 内管壁（Dirichlet）
    ux_bc1, uy_bc1, p_bc1, T_bc1, phi_bc1, rho_x_bc1 = model(x_bc1)

    x_bc2 = bc2_points.requires_grad_(True)  # 外壳（Neumann）
    ux_bc2, uy_bc2, p_bc2, T_bc2, phi_bc2, rho_x_bc2 = model(x_bc2)

    # -------------------------- 2. PDE损失L_PDE（思路3.1）--------------------------
    # 2.1 质量守恒损失L_mass（文档方程1：∇·u=0）
    # 自动微分计算∂ux/∂x, ∂uy/∂y
    grad_ux = torch.autograd.grad(ux_col, x_col, grad_outputs=torch.ones_like(ux_col),
                                  create_graph=True, retain_graph=True)[0]
    dudx = grad_ux[:, 0:1]  # ux对x的偏导
    dudy = grad_ux[:, 1:2]  # ux对y的偏导（无用，仅为了提取维度）

    grad_uy = torch.autograd.grad(uy_col, x_col, grad_outputs=torch.ones_like(uy_col),
                                  create_graph=True, retain_graph=True)[0]
    dvdy = grad_uy[:, 1:2]  # uy对y的偏导
    dvdx = grad_uy[:, 0:1]  # uy对x的偏导（无用）

    mass_residual = dudx + dvdy  # 质量守恒残差：∂ux/∂x + ∂uy/∂y
    L_mass = torch.sum(mass_residual ** 2 * collocation_weights) / torch.sum(collocation_weights)

    # 2.2 动量守恒损失L_momentum（文档方程2-3：x,y方向）
    # 关键项计算：自动微分求一阶/二阶导数
    # 对流项：u·∇ux = ux*∂ux/∂x + uy*∂ux/∂y；u·∇uy = ux*∂uy/∂x + uy*∂uy/∂y
    convect_ux = ux_col * dudx + uy_col * dudy
    convect_uy = ux_col * dvdx + uy_col * dvdy

    # 压力梯度项：-1/ρ * ∂p/∂x；-1/ρ * ∂p/∂y
    grad_p = torch.autograd.grad(p_col, x_col, grad_outputs=torch.ones_like(p_col),
                                 create_graph=True, retain_graph=True)[0]
    dpdx = grad_p[:, 0:1]
    dpdy = grad_p[:, 1:2]
    pressure_term_x = -dpdx / rho_total
    pressure_term_y = -dpdy / rho_total

    # 粘性项：ν∇²ux = ν(∂²ux/∂x² + ∂²ux/∂y²)；ν∇²uy = ν(∂²uy/∂x² + ∂²uy/∂y²)
    # 二阶导数：对一阶导数再求导
    d2udx2 = torch.autograd.grad(dudx, x_col, grad_outputs=torch.ones_like(dudx),
                                 create_graph=True, retain_graph=True)[0][:, 0:1]
    d2udy2 = torch.autograd.grad(dudy, x_col, grad_outputs=torch.ones_like(dudy),
                                 create_graph=True, retain_graph=True)[0][:, 1:2]
    viscous_term_x = nu_col * (d2udx2 + d2udy2)

    d2vdx2 = torch.autograd.grad(dvdx, x_col, grad_outputs=torch.ones_like(dvdx),
                                 create_graph=True, retain_graph=True)[0][:, 0:1]
    d2vdy2 = torch.autograd.grad(dvdy, x_col, grad_outputs=torch.ones_like(dvdy),
                                 create_graph=True, retain_graph=True)[0][:, 1:2]
    viscous_term_y = nu_col * (d2vdx2 + d2vdy2)

    # 糊状区源项：-S_t*ux；-S_t*uy（文档方程2）
    source_term_x = -S_t * ux_col
    source_term_y = -S_t * uy_col

    # 浮力项（仅y方向）：仅作用于液态PCM（铜占比rho_x_col，液态PCM占比(1-rho_x_col)*phi_col）
    F_B = rho_l * alpha * g * (T_col - Tpc) * (1.0 - rho_x_col) * phi_col

    # 动量残差（x,y方向）
    mom_residual_x = convect_ux - (pressure_term_x + viscous_term_x + source_term_x)
    mom_residual_y = convect_uy - (pressure_term_y + viscous_term_y + source_term_y + F_B)
    L_momentum = torch.sum((mom_residual_x ** 2 + mom_residual_y ** 2) * collocation_weights) / torch.sum(collocation_weights)

    # 2.3 传热控制损失L_heat（文档方程6：∂T/∂τ + u·∇T - a∇²T=0）
    # 一阶导数：∂T/∂x, ∂T/∂y, ∂T/∂τ（τ为归一化时间）
    grad_T = torch.autograd.grad(T_col, x_col, grad_outputs=torch.ones_like(T_col),
                             create_graph=True, retain_graph=True)[0]
    dTdx = grad_T[:, 0:1]
    dTdy = grad_T[:, 1:2]
    dTdtau_norm = grad_T[:, 2:3]  # 对归一化时间（τ_norm=τ/1e5）的导数

    # 时间导数量级转换：∂T/∂τ（实际时间）= ∂T/∂τ_norm * dτ_norm/dτ = dTdtau_norm / 1e5
    dTdtau_real = dTdtau_norm / 1e5

    # 对流项：u·∇T = ux*∂T/∂x + uy*∂T/∂y
    convect_T = ux_col * dTdx + uy_col * dTdy

    # 扩散项：a∇²T = a(∂²T/∂x² + ∂²T/∂y²)
    d2Tdx2 = torch.autograd.grad(dTdx, x_col, grad_outputs=torch.ones_like(dTdx),
                                 create_graph=True, retain_graph=True)[0][:, 0:1]
    d2Tdy2 = torch.autograd.grad(dTdy, x_col, grad_outputs=torch.ones_like(dTdy),
                                 create_graph=True, retain_graph=True)[0][:, 1:2]
    diffusive_term = a_col * (d2Tdx2 + d2Tdy2)

    # 传热残差（使用实际时间导数）
    heat_residual = dTdtau_real + convect_T - diffusive_term
    L_heat = torch.sum(heat_residual ** 2 * collocation_weights) / torch.sum(collocation_weights)

    # 2.4 液相率约束损失L_phi（φ∈[0,1]）
    L_phi = torch.sum((torch.max(torch.tensor(0.0).to(device), -phi_col) ** 2 +
                   torch.max(torch.tensor(0.0).to(device), phi_col - 1.0) ** 2) * collocation_weights) / torch.sum(collocation_weights)

    # 压力正则化：避免压力数值过大（小幅惩罚，不影响物理约束）
    L_pressure_reg = torch.mean(p_col ** 2) * 1e-6
    # PDE总损失
    L_PDE = L_mass + L_momentum + L_heat + L_phi + L_pressure_reg

    # -------------------------- 3. 初始/边界条件损失L_IC/BC--------------------------
    # 3.1 初始条件损失L_IC（τ=0）
    T0 = 290.0 if heat_storage else 360.0  # 储热T0=290K，释热T0=360K
    L_IC = torch.sum(((T_ic - T0) ** 2 * 10 + ux_ic ** 2 + uy_ic ** 2) * ic_weights) / torch.sum(ic_weights)

    # 3.2 边界条件损失L_BC
    # 内管壁（Dirichlet）：T=360K（储热）或290K（释热）
    Tw = 360.0 if heat_storage else 290.0
    L_BC1 = torch.mean((T_bc1 - Tw) ** 2 * 2)  # (T_bc1-Tw)²乘以2

    # 外壳（Neumann）：绝热，∂T/∂n=0（法向导数为0）
    grad_T_bc2 = torch.autograd.grad(T_bc2, x_bc2, grad_outputs=torch.ones_like(T_bc2), create_graph=True, retain_graph=True)[0]
    dTdx_bc2 = grad_T_bc2[:, 0:1]
    dTdy_bc2 = grad_T_bc2[:, 1:2]
    x_bc2_real = x_bc2[:, 0:1] * r1
    y_bc2_real = x_bc2[:, 1:2] * r1
    r_bc2 = r1  # 外壳边界的半径固定为r1，直接用常数
    dTdn = (x_bc2_real / r_bc2) * dTdx_bc2 + (y_bc2_real / r_bc2) * dTdy_bc2
    L_BC2 = torch.mean(dTdn**2)

    # IC/BC总损失
    L_IC_BC = L_IC + L_BC1 + L_BC2

    # -------------------------- 4. 拓扑约束损失L_topo-constraint（思路3.3）--------------------------
    # 4.1 ρx取值约束（ρx∈[0,1]）
    L_rho_bounds = torch.mean(torch.max(torch.tensor(0.0).to(device), -rho_x_col) ** 2 +
                              torch.max(torch.tensor(0.0).to(device), rho_x_col - 1.0) ** 2)

    # 4.2 体积比约束（∫ρx dV ≤ φ_total∫dV）
    rho_x_avg = torch.sum(rho_x_col * collocation_weights) / torch.sum(collocation_weights)
    vol_residual = rho_x_avg - phi_total
    # 用logistic函数实现平滑软约束（10是约束强度，可调整）
    L_rho_vol = torch.log(1 + torch.exp(10 * vol_residual)) / 10

    # 拓扑约束总损失
    L_topo_constraint = L_rho_bounds + L_rho_vol

    # -------------------------- 5. 优化目标损失L_objective（思路3.4，分Case）--------------------------
    # 采样优化目标点（设计域内）+ 面积元权重，开启自动求导
    x_obj, weight_obj = sample_collocation_points(N_obj)
    x_obj = x_obj.requires_grad_(True)
    ux_obj, uy_obj, p_obj, T_obj, phi_obj, rho_x_obj = model(x_obj)
    T_obj = torch.clamp(T_obj, 285.0, 365.0)  # 约束温度
    weight_obj = weight_obj.unsqueeze(1)  # [N_obj,1]，匹配张量维度

    # Case1：最小平均温度（文档方程14，加权平均）
    T_ave = torch.sum(T_obj * weight_obj) / torch.sum(weight_obj)
    L_obj1 = T_ave ** 2

    # Case2：最小温度均方差（文档方程15，加权均方差）
    T_var = torch.sum(((T_obj - T_ave) ** 2) * weight_obj) / torch.sum(weight_obj)
    L_obj2 = T_var

    # Case3：多目标（平均温度+温度均方差+火积耗散）
    # 火积耗散（文档方程16：φ_g=∫λ∇²T dA，加权积分）
    grad_T_obj = torch.autograd.grad(T_obj, x_obj, grad_outputs=torch.ones_like(T_obj),
                                     create_graph=True, retain_graph=True)[0]
    dTdx_obj = grad_T_obj[:, 0:1]
    dTdy_obj = grad_T_obj[:, 1:2]
    d2Tdx2_obj = torch.autograd.grad(dTdx_obj, x_obj, grad_outputs=torch.ones_like(dTdx_obj),
                                     create_graph=True, retain_graph=True)[0][:, 0:1]
    d2Tdy2_obj = torch.autograd.grad(dTdy_obj, x_obj, grad_outputs=torch.ones_like(dTdy_obj),
                                     create_graph=True, retain_graph=True)[0][:, 1:2]
    laplacian_T = torch.clamp(d2Tdx2_obj + d2Tdy2_obj, -1e3, 1e3)
    # 计算混合导热系数λ_total_obj
    rho_total_obj, lambda_total_obj, _, _, _, _, _ = compute_thermo_props(phi_obj, rho_x_obj, T_obj)
    phi_g = torch.sum(lambda_total_obj * laplacian_T * weight_obj) / torch.sum(weight_obj)
    L_obj3 = w1 * L_obj1 + w2 * L_obj2 + w3 * phi_g ** 2

    # 选择对应Case的目标损失
    if case == 1:
        L_objective = L_obj1
    elif case == 2:
        L_objective = L_obj2
    else:
        L_objective = L_obj3

    # -------------------------- 6. 总损失 --------------------------
    L_total = lambda1 * L_PDE + lambda2 * L_IC_BC + lambda3 * L_topo_constraint + lambda4 * L_objective

    # 返回各损失项（用于训练监控）
    return L_total, L_PDE, L_IC_BC, L_topo_constraint, L_objective

In [ ]:
def train_model(model, epochs_pre=1000, epochs_fine=2000):
    """训练流程：预训练（满足物理约束）→ 精细优化（融合目标）"""
    #  优化器配置（对应思路5.1-5.2）
    # 预训练：Adam优化器，快速降低物理约束损失
    optimizer_adam = optim.Adam(model.parameters(), lr=2e-4, weight_decay=1e-6)
    scheduler = CosineAnnealingLR(optimizer_adam, T_max=200, eta_min=1e-6)  # T_max=周期，eta_min=最小学习率

    # 精细优化：L-BFGS优化器，高精度梯度优化
    optimizer_lbfgs = optim.LBFGS(model.parameters(), lr=5e-4, max_iter=100, history_size=50)
    
    # 3. 预训练（仅优化物理约束+拓扑约束，屏蔽目标）
    print("="*50)
    print("开始预训练（满足物理约束）")
    print("="*50)
    global lambda4  # 临时修改目标损失权重为0
    lambda4_backup = lambda4
    lambda4 = 0.0

    # 预训练前初始化配点和权重
    collocation_points, collocation_weights = sample_collocation_points(N_mass)
    ic_points, ic_weights = sample_initial_condition(N_IC)
    bc1_points, bc2_points = sample_boundary(N_BC1, N_BC2)

    for epoch in range(epochs_pre):
        # 每50轮进行自适应采样（基于传热残差）
        if epoch % 50 == 0 and epoch != 0:
            # 步骤1：保存模型参数的requires_grad状态（后续恢复）
            param_requires_grad = {param: param.requires_grad for param in model.parameters()}
            # 步骤2：禁用模型参数的梯度（避免追踪参数梯度，但x_col_temp的梯度可以算）
            for param in model.parameters():
                param.requires_grad = False
        
            # 计算上一轮配点的传热残差（用于自适应采样）
            x_col_temp = collocation_points.requires_grad_(True)
            ux_col_temp, uy_col_temp, p_col_temp, T_col_temp, phi_col_temp, rho_x_col_temp = model(x_col_temp)
            rho_total_temp, lambda_total_temp, mu_total_temp, nu_col_temp, cp_total_temp, a_col_temp, S_t_temp = compute_thermo_props(phi_col_temp, rho_x_col_temp, T_col_temp)
            
            # 计算传热残差（仅需数值，临时保留计算图以计算二阶导数）
            grad_T_temp = torch.autograd.grad(
                T_col_temp, x_col_temp, 
                grad_outputs=torch.ones_like(T_col_temp),
                create_graph=True,       # 开启：让一阶导数可求二阶导
                retain_graph=True        # 保留：直到两个二阶导数都计算完成
            )[0]
            dTdx_temp = grad_T_temp[:, 0:1]
            dTdy_temp = grad_T_temp[:, 1:2]
            dTdtau_norm_temp = grad_T_temp[:, 2:3]
            dTdtau_real_temp = dTdtau_norm_temp / 1e5
            convect_T_temp = ux_col_temp * dTdx_temp + uy_col_temp * dTdy_temp
        
            # 计算二阶导数（先保留图，算完第二个再释放）
            d2Tdx2_temp = torch.autograd.grad(
                dTdx_temp, x_col_temp, 
                grad_outputs=torch.ones_like(dTdx_temp),
                create_graph=False,
                retain_graph=True        # 继续保留：给第二个二阶导数用
            )[0][:, 0:1]
            d2Tdy2_temp = torch.autograd.grad(
                dTdy_temp, x_col_temp, 
                grad_outputs=torch.ones_like(dTdy_temp),
                create_graph=False,
                retain_graph=False       # 释放：两个二阶导数都算完了
            )[0][:, 1:2]
        
            diffusive_term_temp = a_col_temp * (d2Tdx2_temp + d2Tdy2_temp)
            heat_residual_temp = dTdtau_real_temp + convect_T_temp - diffusive_term_temp
            # 手动释放计算图（彻底避免残留）
            T_col_temp.detach()
            x_col_temp.requires_grad_(False)  # 关闭x_col_temp的梯度
        
            # 步骤3：恢复模型参数的requires_grad状态（不影响后续训练）
            for param, req_grad in param_requires_grad.items():
                param.requires_grad = req_grad
        
            # 自适应采样更新配点和权重
            collocation_points, collocation_weights = adaptive_sample(model, N_mass, collocation_points, heat_residual_temp)
            # 边界点和初始点固定更新（非自适应）
            bc1_points, bc2_points = sample_boundary(N_BC1, N_BC2)
            ic_points, ic_weights = sample_initial_condition(N_IC)
        # 计算损失（储热过程，可切换为释热）
        L_total, L_PDE, L_IC_BC, L_topo, L_obj = compute_loss(
            model, collocation_points, collocation_weights, ic_points, ic_weights, bc1_points, bc2_points, heat_storage=True
        )

        # 反向传播+参数更新
        L_total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)  # 梯度裁剪防爆炸
        optimizer_adam.step()
        scheduler.step()

        # 每100轮打印损失
        if (epoch + 1) % 100 == 0:
            print(f"预训练Epoch [{epoch+1}/{epochs_pre}] | "
                  f"总损失: {L_total.item():.6f} | "
                  f"PDE损失: {L_PDE.item():.6f} | "
                  f"IC/BC损失: {L_IC_BC.item():.6f} | "
                  f"拓扑约束损失: {L_topo.item():.6f}")

    # 恢复目标损失权重
    lambda4 = lambda4_backup

    # 4. 精细优化（融合优化目标，用L-BFGS）
    print("\n" + "="*50)
    print("开始精细优化（融合优化目标）")
    print("="*50)

    def closure():
        """L-BFGS需要的闭包函数（重复计算损失和梯度）"""
        optimizer_lbfgs.zero_grad()
        L_total, L_PDE, L_IC_BC, L_topo, L_obj = compute_loss(
            model, collocation_points, collocation_weights, ic_points, ic_weights, bc1_points, bc2_points, heat_storage=True
        )
        L_total.backward()
        # 梯度裁剪：避免梯度爆炸，提升L-BFGS稳定性
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
        return L_total

    for epoch in range(epochs_fine):
        optimizer_lbfgs.step(closure)
        # 计算当前损失用于打印
        L_total, L_PDE, L_IC_BC, L_topo, L_obj = compute_loss(
            model, collocation_points, collocation_weights, ic_points, ic_weights, bc1_points, bc2_points, heat_storage=True
        )

        # 每50轮打印损失
        if (epoch + 1) % 50 == 0:
            print(f"精细优化Epoch [{epoch+1}/{epochs_fine}] | "
                  f"总损失: {L_total.item():.6f} | "
                  f"PDE损失: {L_PDE.item():.6f} | "
                  f"目标损失: {L_obj.item():.6f} | "
                  f"体积比: {torch.mean(model(collocation_points)[5]).item():.3f}")

    # 5. 保存训练好的模型
    torch.save(model.state_dict(), f"topo_pinn_case{case}.pth")
    print("\n训练完成！模型已保存为 topo_pinn_case{case}.pth")

def compute_key_metrics(model, heat_storage=True):
    """
    计算论文关键指标，验证模型精度
    返回：完全熔化时间（φ=0.98）、平均储热容量、场协同角均值
    """
    # 1. 完全熔化时间计算（τ从0到1e5s，步长10s）
    complete_melting_time = None
    for tau in range(0, 100000, 10):
        tau_norm = tau / 1e5
        # 生成设计域配点
        collocation_points, collocation_weights = sample_collocation_points(5000)
        x = collocation_points.clone()
        x[:, 2:3] = torch.full_like(x[:, 2:3], tau_norm)
        _, _, _, _, phi, _ = model(x)
        phi_avg = torch.sum(phi * collocation_weights) / torch.sum(collocation_weights)  # 加权平均
        if phi_avg >= 0.98:
            complete_melting_time = tau
            break
    
    # 2. 平均储热容量计算（积分热流率/时间，τ=0到完全熔化时间）
    average_heat_storage = None
    if complete_melting_time is not None:
        heat_flux_list = []
        for tau in range(0, complete_melting_time, 50):
            tau_norm = tau / 1e5
            collocation_points, _ = sample_collocation_points(3000)
            x = collocation_points.clone()
            x[:, 2:3] = torch.full_like(x[:, 2:3], tau_norm)
            x.requires_grad_(True)
            ux, uy, p, T, phi, rho_x = model(x)
            rho_total, lambda_total, _, _, _, a, _ = compute_thermo_props(phi, rho_x, T)
            # 计算热流密度q=-λ∇T（内管壁处）
            grad_T = torch.autograd.grad(T, x, grad_outputs=torch.ones_like(T), create_graph=True, retain_graph=True)[0]
            dTdr = (x[:, 0:1]*r1 * grad_T[:, 0:1] + x[:, 1:2]*r1 * grad_T[:, 1:2]) / (torch.sqrt((x[:, 0:1]*r1)**2 + (x[:, 1:2]*r1)**2) + 1e-10)
            q = -lambda_total * dTdr
            heat_flux_list.append(torch.mean(q).item())
        average_heat_storage = np.mean(heat_flux_list)
    
    # 3. 场协同角均值（τ=200到1200s，传热对流主导阶段）
    field_synergy_angle_avg = None
    if complete_melting_time is not None and complete_melting_time > 1200:
        theta_list = []
        for tau in range(200, 1201, 100):
            tau_norm = tau / 1e5
            collocation_points, _ = sample_collocation_points(3000)
            x = collocation_points.clone()
            x[:, 2:3] = torch.full_like(x[:, 2:3], tau_norm)
            x.requires_grad_(True)
            ux, uy, p, T, phi, rho_x = model(x)
            # 计算u·∇T和||u||·||∇T||
            grad_T = torch.autograd.grad(T, x, grad_outputs=torch.ones_like(T), create_graph=True, retain_graph=True)[0]
            u_dot_gradT = ux * grad_T[:, 0:1] + uy * grad_T[:, 1:2]
            norm_u = torch.sqrt(ux**2 + uy**2 + 1e-10)
            norm_gradT = torch.sqrt(grad_T[:, 0:1]**2 + grad_T[:, 1:2]**2 + 1e-10)
            cos_theta = u_dot_gradT / (norm_u * norm_gradT)
            cos_theta = torch.clamp(cos_theta, -1.0, 1.0)  # 避免数值溢出
            theta = torch.acos(cos_theta) * (180 / np.pi)  # 弧度→角度
            theta_list.append(torch.mean(theta).item())
        field_synergy_angle_avg = np.mean(theta_list)
    
    return {
        "完全熔化时间(s)": complete_melting_time,
        "平均储热容量(W)": average_heat_storage,
        "场协同角均值(°)": field_synergy_angle_avg
    }

def visualize_results(model, save_path="./results"):
    """可视化结果：温度云图、液相率云图、拓扑结构（rho_x）"""
    import os
    os.makedirs(save_path, exist_ok=True)
    
    # 生成环形域网格点
    r = np.linspace(r0, r1, 100)
    theta = np.linspace(0, 2*np.pi, 100)
    R, Theta = np.meshgrid(r, theta)
    X = R * np.cos(Theta)
    Y = R * np.sin(Theta)
    
    # 选择关键时间点可视化（储热过程）
    tau_list = [100, 300, 500, 751, 1000]  # 751s为论文Case3完全熔化时间
    for tau in tau_list:
        tau_norm = tau / 1e5
        # 构造输入（x_norm, y_norm, tau_norm）
        x_input = np.hstack([
            X.reshape(-1, 1)/r1,
            Y.reshape(-1, 1)/r1,
            np.full((10000, 1), tau_norm)
        ])
        x_tensor = torch.tensor(x_input, dtype=torch.float32).to(device)
        ux, uy, p, T, phi, rho_x = model(x_tensor)
        
        # 转换为网格格式
        T_grid = T.detach().cpu().numpy().reshape(100, 100)
        phi_grid = phi.detach().cpu().numpy().reshape(100, 100)
        rho_x_grid = rho_x.detach().cpu().numpy().reshape(100, 100)
        
        # 绘制温度云图
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1)
        contourf = plt.contourf(X, Y, T_grid, cmap='jet', vmin=290, vmax=360)
        plt.colorbar(contourf, label='Temperature (K)')
        plt.title(f'Temperature (τ={tau}s)')
        plt.axis('equal')
        
        # 绘制液相率云图
        plt.subplot(1, 3, 2)
        contourf = plt.contourf(X, Y, phi_grid, cmap='viridis', vmin=0, vmax=1)
        plt.colorbar(contourf, label='Liquid Fraction φ')
        plt.title(f'Liquid Fraction (τ={tau}s)')
        plt.axis('equal')
        
        # 绘制拓扑结构（rho_x≥0.5为铜，否则为PCM）
        plt.subplot(1, 3, 3)
        contourf = plt.contourf(X, Y, (rho_x_grid >= 0.5).astype(int), cmap='binary')
        plt.colorbar(contourf, label='Topology (1=Cu, 0=PCM)')
        plt.title(f'Topological Structure (τ={tau}s)')
        plt.axis('equal')
        
        plt.tight_layout()
        plt.savefig(os.path.join(save_path, f'result_tau_{tau}s.png'), dpi=300, bbox_inches='tight')
        plt.close()
    
    print(f"可视化结果已保存到 {save_path}")

# -------------------------- 启动训练 --------------------------
if __name__ == "__main__":
    # 初始化模型（残差网络：6层隐藏层，256神经元）
    model = TopoPINN(input_dim=3, output_dim=6, hidden_layers=6, hidden_dim=256).to(device)
    # 启动训练（预训练20000轮，精细优化2000轮）
    train_model(model, epochs_pre=20000, epochs_fine=2000)
    
    # 加载训练好的模型（可选，若训练中断可直接加载）
    model.load_state_dict(torch.load(f"topo_pinn_case{case}.pth"))
    model.eval()  # 切换为评估模式
    
    # 计算关键指标，与论文对比
    print("\n" + "="*50)
    print("关键指标验证（对标论文Case3）")
    print("="*50)
    metrics = compute_key_metrics(model, heat_storage=True)
    for key, value in metrics.items():
        if value is not None:
            print(f"{key}: {value:.2f}")
        else:
            print(f"{key}: 未计算成功")
    
    # 可视化结果（温度、液相率、拓扑结构）
    visualize_results(model, save_path="./topo_pinn_results")
    print("\n训练+验证+可视化完成！")

开始预训练（满足物理约束）
预训练Epoch [100/20000] | 总损失: 6038134.000000 | PDE损失: 107.270432 | IC/BC损失: 395.390869 | 拓扑约束损失: 0.052663
预训练Epoch [200/20000] | 总损失: 3568927.500000 | PDE损失: 39.238407 | IC/BC损失: 235.312576 | 拓扑约束损失: 0.047391
预训练Epoch [300/20000] | 总损失: 3831178.250000 | PDE损失: 33.716408 | IC/BC损失: 253.164093 | 拓扑约束损失: 0.043667
预训练Epoch [400/20000] | 总损失: 6441022.500000 | PDE损失: 29.120981 | IC/BC损失: 427.460114 | 拓扑约束损失: 0.042279
